# NN implementation of SR latch using a Hopfield network

In [1]:
import numpy as np
import pandas as pd

### Hopfield network implementation (from scratch)

In [2]:
class SRHopfieldLatch:
    def __init__(self, beta=2.0):
        """
        2-neuron Hopfield model for SR latch:
        neuron 0 -> Q
        neuron 1 -> Q_bar
        """
        self.beta = beta
        
        # Weight matrix that makes [+1, -1] and [-1, +1] stable attractors
        self.W = np.array([
            [0.0, -1.0],
            [-1.0, 0.0]
        ])

    def bias(self, S, R):
        """
        Bias vector as a function of (S, R) in bipolar form.
        Convention:
            Hold   : S=-1, R=-1 -> [0, 0]
            Set    : S=+1, R=-1 -> [+beta, -beta]
            Reset  : S=-1, R=+1 -> [-beta, +beta]
            Invalid: S=+1, R=+1 -> [-beta, -beta]  (NOR-type active-high convention)
        """
        if S == -1 and R == -1:   # Hold
            return np.array([0.0, 0.0])
        elif S == +1 and R == -1: # Set
            return np.array([ self.beta, -self.beta])
        elif S == -1 and R == +1: # Reset
            return np.array([-self.beta,  self.beta])
        elif S == +1 and R == +1: # Invalid
            return np.array([-self.beta, -self.beta])
        else:
            raise ValueError("S and R must each be either -1 or +1")

    def sign(self, x):
        """
        Bipolar sign activation.
        We define sign(0)=+1 for deterministic behavior.
        """
        return 1 if x >= 0 else -1

    def energy(self, x, b):
        """
        Hopfield energy:
            E = -1/2 x^T W x - b^T x
        """
        x = np.array(x, dtype=float)
        return -0.5 * x.T @ self.W @ x - b.T @ x

    def update_async_one_sweep(self, x, b, order=None):
        """
        One asynchronous sweep over all neurons.
        """
        x = x.copy()
        n = len(x)

        if order is None:
            order = range(n)

        for i in order:
            y_i = np.dot(self.W[i], x) + b[i]
            x[i] = self.sign(y_i)

        return x

    def run(self, x0, S, R, max_iter=20, order=None):
        """
        Run until convergence or max_iter sweeps.
        Returns:
            final_state, converged, iterations, trajectory
        """
        x = np.array(x0, dtype=int).copy()
        b = self.bias(S, R)

        trajectory = [x.copy()]
        converged = False

        for it in range(1, max_iter + 1):
            x_new = self.update_async_one_sweep(x, b, order=order)
            trajectory.append(x_new.copy())

            if np.array_equal(x_new, x):
                converged = True
                x = x_new
                break

            x = x_new

        return x, converged, it, trajectory


### Helper functions for interpretation

In [3]:
VALID_STATES = [
    np.array([+1, -1]),
    np.array([-1, +1])
]

INVALID_STATES = [
    np.array([+1, +1]),
    np.array([-1, -1])
]

def state_to_string(x):
    return f"[{x[0]:+d}, {x[1]:+d}]"

def is_valid_state(x):
    return any(np.array_equal(x, v) for v in VALID_STATES)

def latch_operation_name(S, R):
    if S == -1 and R == -1:
        return "Hold"
    elif S == +1 and R == -1:
        return "Set"
    elif S == -1 and R == +1:
        return "Reset"
    elif S == +1 and R == +1:
        return "Invalid"
    return "Unknown"

def expected_state(S, R, initial_state=None):
    """
    Expected output under our chosen convention.
    For Hold:
      - if initial state is valid, expected is to keep the same state
      - otherwise it is undefined/ambiguous as a strict latch memory function
    """
    if S == +1 and R == -1:   # Set
        return np.array([+1, -1])
    elif S == -1 and R == +1: # Reset
        return np.array([-1, +1])
    elif S == +1 and R == +1: # Invalid (NOR convention)
        return np.array([-1, -1])
    elif S == -1 and R == -1: # Hold
        if initial_state is not None and is_valid_state(initial_state):
            return np.array(initial_state)
        else:
            return None
    return None


### Exhaustive test over all initial states and all input combination

Because the network has only 2 neurons, all possible initial states are only:

$
[+1,-1],\;[-1,+1],\;[+1,+1],\;[-1,-1]
$

These already cover:
- clean stored patterns
- one-bit corrupted patterns
- two-bit flips
- invalid states

So exhaustive testing is easy and complete.

In [4]:
# All possible 2-bit bipolar initial states
all_initial_states = [
    np.array([+1, -1]),
    np.array([-1, +1]),
    np.array([+1, +1]),
    np.array([-1, -1]),
]

# All SR input combinations in bipolar form
input_cases = [
    (-1, -1),  # Hold
    (+1, -1),  # Set
    (-1, +1),  # Reset
    (+1, +1),  # Invalid
]

net = SRHopfieldLatch(beta=2.0)

rows = []

for S, R in input_cases:
    op_name = latch_operation_name(S, R)
    b = net.bias(S, R)

    for x0 in all_initial_states:
        final_state, converged, iterations, trajectory = net.run(
            x0=x0, S=S, R=R, max_iter=20, order=[0, 1]
        )

        expected = expected_state(S, R, x0)
        if expected is None:
            task_success = None
        else:
            task_success = np.array_equal(final_state, expected)

        rows.append({
            "Operation": op_name,
            "S": S,
            "R": R,
            "Bias": state_to_string(b.astype(int)),
            "Initial State": state_to_string(x0),
            "Final State": state_to_string(final_state),
            "Converged": converged,
            "Iterations": iterations,
            "Final Valid?": is_valid_state(final_state),
            "Matches Expected?": task_success
        })

results_df = pd.DataFrame(rows)
display(results_df)


,Operation,S,R,Bias,Initial State,Final State,Converged,Iterations,Final Valid?,Matches Expected?
0,Hold,-1,-1,"[+0, +0]","[+1, -1]","[+1, -1]",True,1,True,True
1,Hold,-1,-1,"[+0, +0]","[-1, +1]","[-1, +1]",True,1,True,True
2,Hold,-1,-1,"[+0, +0]","[+1, +1]","[-1, +1]",True,2,True,None
3,Hold,-1,-1,"[+0, +0]","[-1, -1]","[+1, -1]",True,2,True,None
4,Set,1,-1,"[+2, -2]","[+1, -1]","[+1, -1]",True,1,True,True
5,Set,1,-1,"[+2, -2]","[-1, +1]","[+1, -1]",True,2,True,True
6,Set,1,-1,"[+2, -2]","[+1, +1]","[+1, -1]",True,2,True,True
7,Set,1,-1,"[+2, -2]","[-1, -1]","[+1, -1]",True,2,True,True
8,Reset,-1,1,"[-2, +2]","[+1, -1]","[-1, +1]",True,2,True,True
9,Reset,-1,1,"[-2, +2]","[-1, +1]","[-1, +1]",True,1,True,True


### Show trajectories for each test case

In [5]:
for S, R in input_cases:
    print("=" * 80)
    print(f"Operation: {latch_operation_name(S, R)} | S={S:+d}, R={R:+d} | Bias={state_to_string(net.bias(S, R).astype(int))}")
    
    for x0 in all_initial_states:
        final_state, converged, iterations, trajectory = net.run(
            x0=x0, S=S, R=R, max_iter=20, order=[0, 1]
        )

        trajectory_strings = [state_to_string(x.astype(int)) for x in trajectory]
        print(f"Initial: {state_to_string(x0)} -> Final: {state_to_string(final_state)} | "
              f"Converged: {converged} | Iterations: {iterations}")
        print("Trajectory:", " -> ".join(trajectory_strings))
    print()


Operation: Hold | S=-1, R=-1 | Bias=[+0, +0]
Initial: [+1, -1] -> Final: [+1, -1] | Converged: True | Iterations: 1
Trajectory: [+1, -1] -> [+1, -1]
Initial: [-1, +1] -> Final: [-1, +1] | Converged: True | Iterations: 1
Trajectory: [-1, +1] -> [-1, +1]
Initial: [+1, +1] -> Final: [-1, +1] | Converged: True | Iterations: 2
Trajectory: [+1, +1] -> [-1, +1] -> [-1, +1]
Initial: [-1, -1] -> Final: [+1, -1] | Converged: True | Iterations: 2
Trajectory: [-1, -1] -> [+1, -1] -> [+1, -1]

Operation: Set | S=+1, R=-1 | Bias=[+2, -2]
Initial: [+1, -1] -> Final: [+1, -1] | Converged: True | Iterations: 1
Trajectory: [+1, -1] -> [+1, -1]
Initial: [-1, +1] -> Final: [+1, -1] | Converged: True | Iterations: 2
Trajectory: [-1, +1] -> [+1, -1] -> [+1, -1]
Initial: [+1, +1] -> Final: [+1, -1] | Converged: True | Iterations: 2
Trajectory: [+1, +1] -> [+1, -1] -> [+1, -1]
Initial: [-1, -1] -> Final: [+1, -1] | Converged: True | Iterations: 2
Trajectory: [-1, -1] -> [+1, -1] -> [+1, -1]

Operation: Reset 

### Compute success percentages
We compute several useful metrics:
1. Overall convergence percentage
2. Task success percentage for cases where an exact expected output is defined
3. Valid-state percentage: how often the final state is one of the valid memory states

In [6]:
# Overall convergence percentage
overall_convergence_pct = 100 * results_df["Converged"].mean()

# Task success percentage:
# Exclude rows where expected behavior is undefined (e.g. hold starting from invalid/noisy states)
defined_rows = results_df["Matches Expected?"].notna()
task_success_pct = 100 * results_df.loc[defined_rows, "Matches Expected?"].mean()

# Percentage ending in one of the valid SR latch states
valid_final_pct = 100 * results_df["Final Valid?"].mean()

summary_df = pd.DataFrame({
    "Metric": [
        "Overall convergence percentage",
        "Task success percentage (defined cases only)",
        "Final state is valid percentage"
    ],
    "Value (%)": [
        round(overall_convergence_pct, 2),
        round(task_success_pct, 2),
        round(valid_final_pct, 2)
    ]
})

display(summary_df)

,Metric,Value (%)
0,Overall convergence percentage,100.0
1,Task success percentage (defined cases only),100.0
2,Final state is valid percentage,75.0


### Noise-based experiments from stored patterns
This cell explicitly simulates “corrupted versions of stored patterns” by flipping 1 or 2 bits.

In [7]:
def flip_bits(x, indices):
    x_new = x.copy()
    for idx in indices:
        x_new[idx] *= -1
    return x_new

stored_patterns = [
    np.array([+1, -1]),
    np.array([-1, +1])
]

noise_rows = []

for base in stored_patterns:
    for num_flips in [1, 2]:
        if num_flips == 1:
            flip_sets = [[0], [1]]
        else:
            flip_sets = [[0, 1]]

        for flip_set in flip_sets:
            corrupted = flip_bits(base, flip_set)

            for S, R in input_cases:
                final_state, converged, iterations, trajectory = net.run(
                    x0=corrupted, S=S, R=R, max_iter=20, order=[0, 1]
                )

                expected = expected_state(S, R, corrupted)
                if expected is None:
                    task_success = None
                else:
                    task_success = np.array_equal(final_state, expected)

                noise_rows.append({
                    "Base Pattern": state_to_string(base),
                    "Corrupted Input": state_to_string(corrupted),
                    "Number of Flips": num_flips,
                    "Operation": latch_operation_name(S, R),
                    "S": S,
                    "R": R,
                    "Final State": state_to_string(final_state),
                    "Converged": converged,
                    "Iterations": iterations,
                    "Final Valid?": is_valid_state(final_state),
                    "Matches Expected?": task_success
                })

noise_df = pd.DataFrame(noise_rows)
display(noise_df)

,Base Pattern,Corrupted Input,Number of Flips,Operation,S,R,Final State,Converged,Iterations,Final Valid?,Matches Expected?
0,"[+1, -1]","[-1, -1]",1,Hold,-1,-1,"[+1, -1]",True,2,True,None
1,"[+1, -1]","[-1, -1]",1,Set,1,-1,"[+1, -1]",True,2,True,True
2,"[+1, -1]","[-1, -1]",1,Reset,-1,1,"[-1, +1]",True,2,True,True
3,"[+1, -1]","[-1, -1]",1,Invalid,1,1,"[-1, -1]",True,1,False,True
4,"[+1, -1]","[+1, +1]",1,Hold,-1,-1,"[-1, +1]",True,2,True,None
5,"[+1, -1]","[+1, +1]",1,Set,1,-1,"[+1, -1]",True,2,True,True
6,"[+1, -1]","[+1, +1]",1,Reset,-1,1,"[-1, +1]",True,2,True,True
7,"[+1, -1]","[+1, +1]",1,Invalid,1,1,"[-1, -1]",True,2,False,True
8,"[+1, -1]","[-1, +1]",2,Hold,-1,-1,"[-1, +1]",True,1,True,True
9,"[+1, -1]","[-1, +1]",2,Set,1,-1,"[+1, -1]",True,2,True,True


### Report number of successful convergences

In [ ]:
# 1) Total number of noise test cases
total_cases = len(noise_df)

# 2) Number of runs that converged to any stable state
num_converged = int(noise_df["Converged"].sum())

# 3) Number of runs that converged AND matched the expected SR behavior
# Ignore rows where Matches Expected? is undefined (None / NaN)
defined_expected_mask = noise_df["Matches Expected?"].notna()
num_expected_success = int(
    ((noise_df["Converged"] == True) & (noise_df["Matches Expected?"] == True)).sum()
)

# 4) Number of runs that converged AND ended in one of the valid latch states
num_valid_final_success = int(
    ((noise_df["Converged"] == True) & (noise_df["Final Valid?"] == True)).sum()
)

# 5) Percentages
convergence_pct = 100 * num_converged / total_cases
expected_success_pct = 100 * num_expected_success / max(1, defined_expected_mask.sum())
valid_final_pct = 100 * num_valid_final_success / total_cases

summary_counts_df = pd.DataFrame({
    "Metric": [
        "Total noise test cases",
        "Number of converged runs",
        "Number of successful runs (matched expected output)",
        "Number of converged runs ending in valid latch state",
        "Convergence percentage",
        "Expected-output success percentage",
        "Valid-final-state percentage"
    ],
    "Value": [
        total_cases,
        num_converged,
        num_expected_success,
        num_valid_final_success,
        f"{convergence_pct:.2f}%",
        f"{expected_success_pct:.2f}%",
        f"{valid_final_pct:.2f}%"
    ]
})

display(summary_counts_df)

,Metric,Value
0,Total noise test cases,24
1,Number of converged runs,24
2,Number of successful runs (matched expected ou...,20
3,Number of converged runs ending in valid latch...,18
4,Convergence percentage,100.00%
5,Expected-output success percentage,100.00%
6,Valid-final-state percentage,75.00%
